
# OPTATIVO IV — GOBIERNO DE DATOS

## Proyecto: Red Salud Integral Carazo

### Objetivo
Implementar y verificar las políticas establecidas en el Marco de Gobierno de Datos V2 mediante controles programados, auditorías y pruebas funcionales.

### Documento de referencia
Marco de Gobierno de Datos V2 — II Corte.

### Políticas definidas
- POL-01: Cierre obligatorio de citas.
- POL-02: Completitud de registros.
- POL-03: Estandarización.
- POL-04: Control de duplicidad.
- POL-05: Control de valores anómalos.

### Alcance de la implementación
Se programarán y probarán las políticas POL-02, POL-03, POL-04 y POL-05.

POL-01 se documentará como no evaluable con las variables disponibles, debido a que no se dispone de una fecha de cierre de la cita.


In [1]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO IV - GOBIERNO DE DATOS
#
# PASO 1. CONEXIÓN Y REGISTRO DE POLÍTICAS
# ============================================================

import sys
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv
from sqlalchemy import text
from IPython.display import display


# ============================================================
# 1. LOCALIZAR EL PROYECTO
# ============================================================

rutas = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "RedSaludCarazo_proyecto_base"
]

proyecto = next(
    (
        ruta for ruta in rutas
        if (ruta / "src" / "database.py").is_file()
    ),
    None
)

if proyecto is None:
    raise FileNotFoundError(
        "No se encontró la carpeta del proyecto."
    )

print("Proyecto:", proyecto)


# ============================================================
# 2. CONECTAR CON SQL SERVER
# ============================================================

sys.path.insert(
    0,
    str(proyecto / "src")
)

load_dotenv(
    proyecto / ".env",
    override=True
)

from database import engine_sqlserver

engine = engine_sqlserver()


# ============================================================
# 3. CONSULTAR LOS REGISTROS ORIGINALES
# ============================================================

with engine.connect() as conexion:

    nombre_bd = conexion.execute(
        text("SELECT DB_NAME()")
    ).scalar_one()

    datos_gobierno = pd.read_sql_query(
        text("SELECT * FROM dbo.raw_cita"),
        conexion
    )


# ============================================================
# 4. CREAR CARPETA DE GOBIERNO DE DATOS
# ============================================================

carpeta_gobierno = (
    proyecto
    / "resultados_jupyter"
    / "optativo_IV"
)

carpeta_politicas = (
    carpeta_gobierno / "politicas"
)

carpeta_politicas.mkdir(
    parents=True,
    exist_ok=True
)

fecha = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")


# ============================================================
# 5. MATRIZ DE POLÍTICAS DEL II CORTE
# ============================================================

matriz_politicas = pd.DataFrame([
    {
        "Codigo": "POL-01",
        "Politica": "Cierre obligatorio de citas",
        "Regla": (
            "Consolidar el estado de asistencia "
            "en un máximo de 24 horas"
        ),
        "Responsable": "Supervisores de admisión",
        "Estado": "NO EVALUABLE",
        "Observacion": (
            "No existe fecha de cierre para "
            "verificar el plazo"
        )
    },
    {
        "Codigo": "POL-02",
        "Politica": "Completitud de registros",
        "Regla": (
            "Todos los campos obligatorios "
            "deben contener información"
        ),
        "Responsable": "Admisión y TI",
        "Estado": "POR EVALUAR",
        "Observacion": (
            "Verificar campos requeridos"
        )
    },
    {
        "Codigo": "POL-03",
        "Politica": "Estandarización",
        "Regla": (
            "Utilizar las categorías "
            "autorizadas del sistema"
        ),
        "Responsable": "Jefaturas de servicio",
        "Estado": "POR EVALUAR",
        "Observacion": (
            "Los catálogos actuales son provisionales"
        )
    },
    {
        "Codigo": "POL-04",
        "Politica": "Control de duplicidad",
        "Regla": (
            "No permitir registros de citas duplicados"
        ),
        "Responsable": "Líder de TI",
        "Estado": "POR EVALUAR",
        "Observacion": (
            "Verificar identificadores y registros repetidos"
        )
    },
    {
        "Codigo": "POL-05",
        "Politica": "Control de valores anómalos",
        "Regla": (
            "Las variables numéricas deben "
            "cumplir rangos lógicos válidos"
        ),
        "Responsable": "TI",
        "Estado": "POR EVALUAR",
        "Observacion": (
            "Detectar valores negativos o inválidos"
        )
    }
])


# ============================================================
# 6. PRESENTAR RESULTADOS
# ============================================================

print("\n" + "=" * 70)
print("OPTATIVO IV - GOBIERNO DE DATOS")
print("PASO 1. VERIFICACIÓN DE LA FUENTE")
print("=" * 70)

print("\nBase de datos:", nombre_bd)
print("Tabla: dbo.raw_cita")
print("Registros:", len(datos_gobierno))
print("Columnas:", len(datos_gobierno.columns))

print("\n" + "=" * 70)
print("POLÍTICAS DEL MARCO DE GOBIERNO V2")
print("=" * 70)

display(matriz_politicas)


# ============================================================
# 7. VERIFICAR POLÍTICAS
# ============================================================

politicas_esperadas = {
    "POL-01",
    "POL-02",
    "POL-03",
    "POL-04",
    "POL-05"
}

assert (
    set(matriz_politicas["Codigo"])
    == politicas_esperadas
), "Faltan políticas del Marco V2."

assert (
    nombre_bd == "RedSaludCarazoDW_V2"
), "La conexión apunta a otra base de datos."

print("\nMATRIZ DE POLÍTICAS: VERIFICADA")


# ============================================================
# 8. GUARDAR EVIDENCIA
# ============================================================

ruta = (
    carpeta_politicas
    / f"matriz_politicas_v2_{fecha}.csv"
)

matriz_politicas.to_csv(
    ruta,
    index=False,
    encoding="utf-8-sig"
)

print("\nEvidencia guardada:")
print(ruta)

print("\nPASO 1 FINALIZADO")
print("No se modificaron registros de SQL Server.")


Proyecto: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base

OPTATIVO IV - GOBIERNO DE DATOS
PASO 1. VERIFICACIÓN DE LA FUENTE

Base de datos: RedSaludCarazoDW_V2
Tabla: dbo.raw_cita
Registros: 10080
Columnas: 19

POLÍTICAS DEL MARCO DE GOBIERNO V2


,Codigo,Politica,Regla,Responsable,Estado,Observacion
0,POL-01,Cierre obligatorio de citas,Consolidar el estado de asistencia en un máxim...,Supervisores de admisión,NO EVALUABLE,No existe fecha de cierre para verificar el plazo
1,POL-02,Completitud de registros,Todos los campos obligatorios deben contener i...,Admisión y TI,POR EVALUAR,Verificar campos requeridos
2,POL-03,Estandarización,Utilizar las categorías autorizadas del sistema,Jefaturas de servicio,POR EVALUAR,Los catálogos actuales son provisionales
3,POL-04,Control de duplicidad,No permitir registros de citas duplicados,Líder de TI,POR EVALUAR,Verificar identificadores y registros repetidos
4,POL-05,Control de valores anómalos,Las variables numéricas deben cumplir rangos l...,TI,POR EVALUAR,Detectar valores negativos o inválidos



MATRIZ DE POLÍTICAS: VERIFICADA

Evidencia guardada:
c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\optativo_IV\politicas\matriz_politicas_v2_20261024T211513Z.csv

PASO 1 FINALIZADO
No se modificaron registros de SQL Server.



## PASO 2. Programación de las políticas de Gobierno de Datos

### Objetivo
Implementar mediante Python las reglas establecidas en el Marco de Gobierno de Datos V2.

### Políticas evaluadas
- POL-02: Completitud de registros.
- POL-03: Estandarización de categorías.
- POL-04: Control de duplicidad.
- POL-05: Control de valores anómalos.

### Criterios de evaluación
Cada política verifica los registros originales e identifica los incumplimientos mediante el identificador del registro, el campo afectado y la regla aplicada.

### Alcance
Se utiliza la tabla dbo.raw_cita de SQL Server.

Los catálogos de referencia son provisionales y requieren validación institucional.

La evaluación no modifica los datos originales.


In [2]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO IV - GOBIERNO DE DATOS
# PASO 2. PROGRAMACIÓN DE POLÍTICAS
# ============================================================

import re
import unicodedata
import pandas as pd

from datetime import datetime, timezone
from IPython.display import display


# ============================================================
# 1. PREPARAR FUENTE Y CARPETA DE EVIDENCIAS
# ============================================================

datos_politicas = datos_gobierno.copy()

carpeta_programacion = (
    proyecto
    / "resultados_jupyter"
    / "optativo_IV"
    / "politicas"
    / "ejecucion"
)

carpeta_programacion.mkdir(
    parents=True,
    exist_ok=True
)

fecha_programacion = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")


# ============================================================
# 2. DEFINIR REGLAS DEL MARCO V2
# ============================================================

# POL-02: Campos obligatorios utilizados
# en las validaciones del proyecto.

campos_obligatorios_gob = [
    "id_cita",
    "fecha_cita",
    "centro_salud",
    "especialidad",
    "canal_agendamiento",
    "grupo_edad",
    "espera_dias",
    "citas_previas_perdidas",
    "recordatorios_enviados",
    "no_asistio"
]

# POL-03: Catálogos provisionales de referencia.
# Todavía no representan una aprobación institucional.

catalogos_gob = {
    "centro_salud": [
        "JINOTEPE",
        "DIRIAMBA",
        "SAN MARCOS",
        "DOLORES"
    ],
    "especialidad": [
        "MEDICINA GENERAL",
        "LABORATORIO",
        "GINECOLOGÍA",
        "PEDIATRÍA",
        "CARDIOLOGÍA"
    ],
    "canal_agendamiento": [
        "CALL CENTER",
        "WEB",
        "VENTANILLA",
        "REFERENCIA MÉDICA"
    ]
}

# POL-04: Campos originales que determinan
# si dos filas contienen exactamente los mismos datos.

columnas_cita_gob = [
    "id_cita",
    "fecha_cita",
    "centro_salud",
    "canal_agendamiento",
    "grupo_edad",
    "especialidad",
    "costo_estimado_usd",
    "espera_dias",
    "recordatorios_enviados",
    "citas_previas_perdidas",
    "tipo_cobertura",
    "sistema_origen",
    "no_asistio",
    "consentimiento_contacto",
    "fecha_carga"
]

# POL-05: Variables sujetas a validación numérica.

campos_numericos_gob = [
    "espera_dias",
    "recordatorios_enviados",
    "citas_previas_perdidas"
]


# ============================================================
# 3. FUNCIONES REUTILIZABLES
# ============================================================

def detectar_vacios_gob(serie):
    """Detecta nulos, cadenas vacías y espacios."""

    return (
        serie.isna()
        | serie.astype("string")
          .str.strip()
          .eq("")
          .fillna(False)
    )


def normalizar_texto_gob(valor):
    """Normaliza texto para comparar categorías."""

    if pd.isna(valor):
        return ""

    texto = str(valor).strip().upper()

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    texto = "".join(
        caracter
        for caracter in unicodedata.normalize(
            "NFD",
            texto
        )
        if unicodedata.category(caracter) != "Mn"
    )

    return texto


def evaluar_politicas_gobierno(datos):
    """
    Evalúa las políticas POL-02, POL-03,
    POL-04 y POL-05.

    Recibe un DataFrame y devuelve:
    - Detalle de incumplimientos.
    - Resumen por política.

    No modifica el DataFrame recibido.
    """

    df = datos.copy()

    columnas_necesarias = set(
        campos_obligatorios_gob
        + columnas_cita_gob
        + ["id_raw"]
    )

    ausentes = columnas_necesarias - set(df.columns)

    if ausentes:
        raise ValueError(
            f"Faltan columnas: {sorted(ausentes)}"
        )

    if not df["id_raw"].is_unique:
        raise ValueError(
            "La columna id_raw debe ser única."
        )

    hallazgos = []

    def registrar(mascara, politica, tipo,
                 campo, regla):

        afectados = df.loc[mascara]

        for _, fila in afectados.iterrows():

            valor = (
                fila[campo]
                if campo in df.columns
                else "FILA COMPLETA"
            )

            hallazgos.append({
                "id_raw": fila["id_raw"],
                "id_cita": fila["id_cita"],
                "Politica": politica,
                "Tipo_hallazgo": tipo,
                "Campo": campo,
                "Valor_original": (
                    "<NULO>"
                    if pd.isna(valor)
                    else str(valor)
                ),
                "Regla": regla,
                "Estado": "INCUMPLE"
            })

    # ========================================================
    # POL-02. COMPLETITUD
    # ========================================================

    for campo in campos_obligatorios_gob:

        vacios = detectar_vacios_gob(
            df[campo]
        )

        registrar(
            vacios,
            "POL-02",
            "CAMPO_OBLIGATORIO_FALTANTE",
            campo,
            "Los campos requeridos deben contener información"
        )

    # ========================================================
    # POL-03. ESTANDARIZACIÓN
    # ========================================================

    for campo, permitidos in catalogos_gob.items():

        serie = df[campo].astype("string")

        vacios = detectar_vacios_gob(serie)

        equivalencias = {
            normalizar_texto_gob(valor): valor
            for valor in permitidos
        }

        claves = serie.map(
            normalizar_texto_gob
        )

        esperados = claves.map(
            equivalencias
        )

        fuera_catalogo = (
            ~vacios
            & ~claves.isin(equivalencias)
        ).fillna(False)

        registrar(
            fuera_catalogo,
            "POL-03",
            "FUERA_DE_CATALOGO",
            campo,
            "Debe pertenecer al catálogo de referencia"
        )

        formato_diferente = (
            ~vacios
            & claves.isin(equivalencias)
            & serie.ne(
                esperados.astype("string")
            ).fillna(False)
        )

        registrar(
            formato_diferente,
            "POL-03",
            "FORMATO_NO_ESTANDAR",
            campo,
            "Debe respetar el formato de referencia"
        )

    # ========================================================
    # POL-04. CONTROL DE DUPLICIDAD
    # ========================================================

    ids = (
        df["id_cita"]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
    )

    ids_repetidos = (
        ids.notna()
        & ids.duplicated(keep=False)
    )

    registrar(
        ids_repetidos,
        "POL-04",
        "IDENTIFICADOR_REPETIDO",
        "id_cita",
        "El identificador de cita debe ser único"
    )

    filas_repetidas = df.duplicated(
        subset=columnas_cita_gob,
        keep=False
    )

    registrar(
        filas_repetidas,
        "POL-04",
        "FILA_COMPLETAMENTE_DUPLICADA",
        "FILA_COMPLETA",
        "No deben existir filas idénticas"
    )

    # ========================================================
    # POL-05. VALORES ANÓMALOS
    # ========================================================

    for campo in campos_numericos_gob:

        serie = df[campo]

        vacios = detectar_vacios_gob(serie)

        numeros = pd.to_numeric(
            serie,
            errors="coerce"
        )

        invalidos = (
            ~vacios
            & (
                numeros.isna()
                | numeros.lt(0)
                | numeros.mod(1).ne(0)
            )
        ).fillna(False)

        registrar(
            invalidos,
            "POL-05",
            "VALOR_NUMERICO_INVALIDO",
            campo,
            "Debe ser un número entero no negativo"
        )

    # La variable no_asistio acepta 0 y 1.
    # Los faltantes se registran mediante POL-02.

    asistencia = (
        df["no_asistio"]
        .astype("string")
        .str.strip()
    )

    asistencia_invalida = (
        ~detectar_vacios_gob(df["no_asistio"])
        & ~asistencia.isin(["0", "1"])
    ).fillna(False)

    registrar(
        asistencia_invalida,
        "POL-05",
        "RESULTADO_INVALIDO",
        "no_asistio",
        "Solo se admiten los valores 0 y 1"
    )

    # ========================================================
    # CONSTRUIR RESULTADOS
    # ========================================================

    columnas = [
        "id_raw",
        "id_cita",
        "Politica",
        "Tipo_hallazgo",
        "Campo",
        "Valor_original",
        "Regla",
        "Estado"
    ]

    detalle = pd.DataFrame(
        hallazgos,
        columns=columnas
    )

    resumen = (
        detalle.groupby("Politica")
        .agg(
            Incidencias=("id_raw", "size"),
            Registros_afectados=("id_raw", "nunique")
        )
        .reindex(
            ["POL-02", "POL-03", "POL-04", "POL-05"],
            fill_value=0
        )
        .reset_index()
    )

    return detalle, resumen


# ============================================================
# 4. EJECUTAR LAS POLÍTICAS
# ============================================================

detalle_gobierno, resumen_gobierno = (
    evaluar_politicas_gobierno(datos_politicas)
)

print("=" * 70)
print("OPTATIVO IV - GOBIERNO DE DATOS")
print("PASO 2. POLÍTICAS PROGRAMADAS")
print("=" * 70)

print("\nRegistros evaluados:", len(datos_politicas))
print("Políticas ejecutadas: 4")

print("\n1. RESULTADOS POR POLÍTICA")
display(resumen_gobierno)


# ============================================================
# 5. DETALLE DE LOS TIPOS DE HALLAZGO
# ============================================================

detalle_tipos_gob = (
    detalle_gobierno
    .groupby(["Politica", "Tipo_hallazgo"])
    .agg(
        Incidencias=("id_raw", "size"),
        Registros_afectados=("id_raw", "nunique")
    )
    .reset_index()
)

print("\n2. RESULTADOS POR TIPO DE HALLAZGO")
display(detalle_tipos_gob)


# ============================================================
# 6. MOSTRAR EJEMPLOS IDENTIFICABLES
# ============================================================

print("\n3. EJEMPLOS DE REGISTROS QUE INCUMPLEN")
display(detalle_gobierno.head(15))


# ============================================================
# 7. COMPROBAR CONSISTENCIA
# ============================================================

assert (
    resumen_gobierno["Incidencias"].sum()
    == len(detalle_gobierno)
), "El resumen no coincide con el detalle."

assert (
    set(resumen_gobierno["Politica"])
    == {"POL-02", "POL-03", "POL-04", "POL-05"}
), "Falta una política."

print("\nCONTROLES DE CONSISTENCIA: SUPERADOS")


# ============================================================
# 8. GUARDAR EVIDENCIAS
# ============================================================

archivos_gob = {
    "ejecucion_resumen_politicas": resumen_gobierno,
    "ejecucion_tipos_hallazgo": detalle_tipos_gob,
    "ejecucion_detalle_politicas": detalle_gobierno
}

for nombre, tabla in archivos_gob.items():

    ruta = (
        carpeta_programacion
        / f"{nombre}_{fecha_programacion}.csv"
    )

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", ruta.name)

print("\nPASO 2 FINALIZADO")
print("No se modificaron los registros de SQL Server.")


OPTATIVO IV - GOBIERNO DE DATOS
PASO 2. POLÍTICAS PROGRAMADAS

Registros evaluados: 10080
Políticas ejecutadas: 4

1. RESULTADOS POR POLÍTICA


,Politica,Incidencias,Registros_afectados
0,POL-02,920,920
1,POL-03,273,273
2,POL-04,254,160
3,POL-05,3,3



2. RESULTADOS POR TIPO DE HALLAZGO


,Politica,Tipo_hallazgo,Incidencias,Registros_afectados
0,POL-02,CAMPO_OBLIGATORIO_FALTANTE,920,920
1,POL-03,FORMATO_NO_ESTANDAR,273,273
2,POL-04,FILA_COMPLETAMENTE_DUPLICADA,94,94
3,POL-04,IDENTIFICADOR_REPETIDO,160,160
4,POL-05,VALOR_NUMERICO_INVALIDO,3,3



3. EJEMPLOS DE REGISTROS QUE INCUMPLEN


,id_raw,id_cita,Politica,Tipo_hallazgo,Campo,Valor_original,Regla,Estado
0,38,RSV2-000038,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,Los campos requeridos deben contener información,INCUMPLE
1,330,RSV2-000330,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,Los campos requeridos deben contener información,INCUMPLE
2,511,RSV2-000511,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,Los campos requeridos deben contener información,INCUMPLE
3,556,RSV2-000556,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,Los campos requeridos deben contener información,INCUMPLE
4,673,RSV2-000673,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,Los campos requeridos deben contener información,INCUMPLE
5,675,RSV2-000675,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,Los campos requeridos deben contener información,INCUMPLE
6,680,RSV2-000680,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,Los campos requeridos deben contener información,INCUMPLE
7,683,RSV2-000683,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,Los campos requeridos deben contener información,INCUMPLE
8,766,RSV2-000766,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,Los campos requeridos deben contener información,INCUMPLE
9,795,RSV2-000795,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,Los campos requeridos deben contener información,INCUMPLE



CONTROLES DE CONSISTENCIA: SUPERADOS
Guardado: ejecucion_resumen_politicas_20261024T211829Z.csv
Guardado: ejecucion_tipos_hallazgo_20261024T211829Z.csv
Guardado: ejecucion_detalle_politicas_20261024T211829Z.csv

PASO 2 FINALIZADO
No se modificaron los registros de SQL Server.



## PASO 3. Auditoría de las políticas de Gobierno de Datos

### Objetivo
Documentar individualmente los incumplimientos detectados por las políticas POL-02, POL-03, POL-04 y POL-05.

### Evidencias
- Auditoría general de incumplimientos.
- Auditoría individual por política.
- Identificación de registros con y sin hallazgos.
- Listado de identificadores repetidos.
- Listado de filas completamente duplicadas.
- Registro de acciones de revisión pendientes.

### Trazabilidad
Cada hallazgo se relaciona con su identificador original y los datos de procedencia almacenados en SQL Server.

La auditoría no modifica ni elimina registros originales.


In [3]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO IV - GOBIERNO DE DATOS
#
# PASO 3. AUDITORÍA DETALLADA DE POLÍTICAS
# ============================================================

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display


# ============================================================
# 1. PREPARAR LOS DATOS
# ============================================================

origen_auditoria = datos_gobierno.copy()
auditoria = detalle_gobierno.copy()

carpeta_auditoria_gob = (
    proyecto
    / "resultados_jupyter"
    / "optativo_IV"
    / "auditoria"
)

carpeta_auditoria_gob.mkdir(
    parents=True,
    exist_ok=True
)

fecha_auditoria = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

if not origen_auditoria["id_raw"].is_unique:
    raise ValueError(
        "Los identificadores id_raw deben ser únicos."
    )


# ============================================================
# 2. AGREGAR INFORMACIÓN DE TRAZABILIDAD
# ============================================================

columnas_trazabilidad = [
    "id_raw",
    "origen_dataset",
    "lote_carga",
    "fecha_ingesta"
]

trazabilidad = origen_auditoria[
    columnas_trazabilidad
].copy()

auditoria = auditoria.merge(
    trazabilidad,
    on="id_raw",
    how="left",
    validate="many_to_one"
)

auditoria.insert(
    0,
    "Codigo_incidencia",
    [
        f"INC-{i:05d}"
        for i in range(1, len(auditoria) + 1)
    ]
)

auditoria["Fecha_auditoria_UTC"] = (
    datetime.now(timezone.utc).isoformat()
)

auditoria["Estado_revision"] = "PENDIENTE"

# Acciones sugeridas según la política.
# No se consideran correcciones ejecutadas.

acciones = {
    "POL-02": "Verificar campos faltantes en la fuente",
    "POL-03": "Revisar equivalencia con el catálogo",
    "POL-04": "Revisar registros del mismo identificador",
    "POL-05": "Verificar el valor numérico original"
}

auditoria["Accion_propuesta"] = (
    auditoria["Politica"].map(acciones)
)


# ============================================================
# 3. RESUMEN GENERAL DE AUDITORÍA
# ============================================================

resumen_auditoria_gob = (
    auditoria
    .groupby("Politica")
    .agg(
        Incidencias=("id_raw", "size"),
        Registros_afectados=("id_raw", "nunique")
    )
    .reindex(
        ["POL-02", "POL-03", "POL-04", "POL-05"],
        fill_value=0
    )
    .reset_index()
)

tipos_auditoria_gob = (
    auditoria
    .groupby(["Politica", "Tipo_hallazgo"])
    .agg(
        Incidencias=("id_raw", "size"),
        Registros_afectados=("id_raw", "nunique")
    )
    .reset_index()
)


# ============================================================
# 4. ESTADO INDIVIDUAL DE TODOS LOS REGISTROS
# ============================================================

# También se documentan los registros sin hallazgos.

incidencias_por_registro = (
    auditoria
    .groupby("id_raw")
    .agg(
        Total_incidencias=("Politica", "size"),
        Politicas_incumplidas=(
            "Politica",
            lambda valores: ", ".join(
                sorted(set(valores))
            )
        )
    )
    .reset_index()
)

estado_registros_gob = origen_auditoria[
    [
        "id_raw",
        "id_cita",
        "origen_dataset",
        "lote_carga",
        "fecha_ingesta"
    ]
].merge(
    incidencias_por_registro,
    on="id_raw",
    how="left",
    validate="one_to_one"
)

estado_registros_gob["Total_incidencias"] = (
    estado_registros_gob["Total_incidencias"]
    .fillna(0)
    .astype(int)
)

estado_registros_gob["Politicas_incumplidas"] = (
    estado_registros_gob["Politicas_incumplidas"]
    .fillna("NINGUNA")
)

estado_registros_gob["Estado_auditoria"] = (
    estado_registros_gob["Total_incidencias"]
    .apply(
        lambda n: (
            "CON_HALLAZGOS"
            if n > 0
            else "SIN_HALLAZGOS"
        )
    )
)


# ============================================================
# 5. AUDITORÍA ESPECIAL DE POL-04: DUPLICIDAD
# ============================================================

# A. Identificadores repetidos

ids_normalizados = (
    origen_auditoria["id_cita"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

frecuencias = ids_normalizados.dropna().value_counts()

grupos_repetidos = frecuencias[
    frecuencias > 1
]

mascara_ids_repetidos = ids_normalizados.isin(
    grupos_repetidos.index
)

detalle_ids_repetidos = origen_auditoria.loc[
    mascara_ids_repetidos,
    [
        "id_raw",
        "id_cita",
        "fecha_cita",
        "centro_salud",
        "no_asistio",
        "lote_carga"
    ]
].copy()

detalle_ids_repetidos["Frecuencia_ID"] = (
    ids_normalizados.loc[
        mascara_ids_repetidos
    ].map(frecuencias).to_numpy()
)

detalle_ids_repetidos = (
    detalle_ids_repetidos
    .sort_values(["id_cita", "id_raw"])
)


# B. Filas completamente duplicadas

mascara_duplicadas = origen_auditoria.duplicated(
    subset=columnas_cita_gob,
    keep=False
)

mascara_copias_adicionales = (
    origen_auditoria.duplicated(
        subset=columnas_cita_gob,
        keep="first"
    )
)

detalle_filas_duplicadas = (
    origen_auditoria.loc[
        mascara_duplicadas
    ]
    .sort_values("id_raw")
    .copy()
)

detalle_filas_duplicadas["Grupo_duplicado"] = (
    detalle_filas_duplicadas
    .groupby(
        columnas_cita_gob,
        dropna=False,
        sort=False
    )
    .ngroup() + 1
)

detalle_filas_duplicadas[
    "Tipo_participacion"
] = "FILA_DEL_GRUPO_DUPLICADO"

total_grupos_ids = len(grupos_repetidos)

total_filas_ids = len(detalle_ids_repetidos)

total_filas_duplicadas = len(
    detalle_filas_duplicadas
)

copias_adicionales = int(
    mascara_copias_adicionales.sum()
)


# ============================================================
# 6. VERIFICACIONES DE CONSISTENCIA
# ============================================================

assert (
    len(estado_registros_gob)
    == len(origen_auditoria)
), "La auditoría no incluye todos los registros."

assert (
    estado_registros_gob["id_raw"].nunique()
    == len(origen_auditoria)
), "Hay identificadores faltantes."

assert (
    resumen_auditoria_gob["Incidencias"].sum()
    == len(auditoria)
), "No coincide el conteo de incidencias."

assert (
    total_filas_ids
    == resumen_auditoria_gob.loc[
        resumen_auditoria_gob["Politica"] == "POL-04",
        "Registros_afectados"
    ].iloc[0]
), "Revisar la auditoría de duplicidad."


# ============================================================
# 7. MOSTRAR RESULTADOS
# ============================================================

print("=" * 70)
print("OPTATIVO IV - GOBIERNO DE DATOS")
print("PASO 3. AUDITORÍA DETALLADA")
print("=" * 70)

print("\n1. RESUMEN GENERAL")
display(resumen_auditoria_gob)

print("\n2. TIPOS DE INCUMPLIMIENTO")
display(tipos_auditoria_gob)

print("\n3. ESTADO DE LOS REGISTROS")

display(
    estado_registros_gob[
        "Estado_auditoria"
    ].value_counts().rename_axis(
        "Estado"
    ).reset_index(name="Cantidad")
)

print("\n4. EJEMPLOS DE HALLAZGOS")
display(auditoria.head(10))

print("\n5. IDENTIFICADORES REPETIDOS")

print("Grupos repetidos:", total_grupos_ids)
print("Filas involucradas:", total_filas_ids)

display(detalle_ids_repetidos.head(10))

print("\n6. FILAS COMPLETAMENTE DUPLICADAS")

print(
    "Filas involucradas:",
    total_filas_duplicadas
)

print(
    "Copias adicionales:",
    copias_adicionales
)

display(
    detalle_filas_duplicadas[
        [
            "Grupo_duplicado",
            "id_raw",
            "id_cita",
            "fecha_cita",
            "centro_salud",
            "no_asistio"
        ]
    ].head(10)
)

print("\nCONTROLES DE AUDITORÍA: SUPERADOS")


# ============================================================
# 8. GUARDAR LOS ARCHIVOS DE AUDITORÍA
# ============================================================

archivos_auditoria = {
    "auditoria_general": auditoria,
    "resumen_auditoria": resumen_auditoria_gob,
    "tipos_hallazgo": tipos_auditoria_gob,
    "estado_todos_los_registros": estado_registros_gob,
    "identificadores_repetidos": detalle_ids_repetidos,
    "filas_duplicadas": detalle_filas_duplicadas
}

for nombre, tabla in archivos_auditoria.items():

    ruta = (
        carpeta_auditoria_gob
        / f"{nombre}_{fecha_auditoria}.csv"
    )

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", ruta.name)


# ============================================================
# 9. EVIDENCIAS INDIVIDUALES POR POLÍTICA
# ============================================================

for codigo in [
    "POL-02",
    "POL-03",
    "POL-04",
    "POL-05"
]:

    detalle = auditoria.loc[
        auditoria["Politica"] == codigo
    ].copy()

    ruta = (
        carpeta_auditoria_gob
        / f"{codigo}_detalle_{fecha_auditoria}.csv"
    )

    detalle.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Guardado:",
        ruta.name,
        "| Incidencias:",
        len(detalle)
    )


print("\nPASO 3 FINALIZADO")
print("Carpeta:", carpeta_auditoria_gob)
print("SQL Server permanece sin modificaciones.")


OPTATIVO IV - GOBIERNO DE DATOS
PASO 3. AUDITORÍA DETALLADA

1. RESUMEN GENERAL


,Politica,Incidencias,Registros_afectados
0,POL-02,920,920
1,POL-03,273,273
2,POL-04,254,160
3,POL-05,3,3



2. TIPOS DE INCUMPLIMIENTO


,Politica,Tipo_hallazgo,Incidencias,Registros_afectados
0,POL-02,CAMPO_OBLIGATORIO_FALTANTE,920,920
1,POL-03,FORMATO_NO_ESTANDAR,273,273
2,POL-04,FILA_COMPLETAMENTE_DUPLICADA,94,94
3,POL-04,IDENTIFICADOR_REPETIDO,160,160
4,POL-05,VALOR_NUMERICO_INVALIDO,3,3



3. ESTADO DE LOS REGISTROS


...


5. IDENTIFICADORES REPETIDOS
Grupos repetidos: 80
Filas involucradas: 160


,id_raw,id_cita,fecha_cita,centro_salud,no_asistio,lote_carga,Frecuencia_ID
91,92,RSV2-000092,2026-04-29,DOLORES,0,20260928T002744032220Z,2
10012,10013,RSV2-000092,2026-04-29,DOLORES,0,20260928T002744032220Z,2
167,168,RSV2-000168,2026-03-20,JINOTEPE,0,20260928T002744032220Z,2
10054,10055,RSV2-000168,2026-03-20,Jinotepe,0,20260928T002744032220Z,2
475,476,RSV2-000476,2026-04-23,SAN MARCOS,0,20260928T002744032220Z,2
10031,10032,RSV2-000476,2026-04-23,SAN MARCOS,0,20260928T002744032220Z,2
662,663,RSV2-000663,2026-04-10,JINOTEPE,1,20260928T002744032220Z,2
10072,10073,RSV2-000663,2026-04-10,Jinotepe,1,20260928T002744032220Z,2
825,826,RSV2-000826,2026-01-20,SAN MARCOS,0,20260928T002744032220Z,2
10041,10042,RSV2-000826,2026-01-20,SAN MARCOS,0,20260928T002744032220Z,2



6. FILAS COMPLETAMENTE DUPLICADAS
Filas involucradas: 94
Copias adicionales: 47


,Grupo_duplicado,id_raw,id_cita,fecha_cita,centro_salud,no_asistio
91,1,92,RSV2-000092,2026-04-29,DOLORES,0
475,2,476,RSV2-000476,2026-04-23,SAN MARCOS,0
825,3,826,RSV2-000826,2026-01-20,SAN MARCOS,0
1439,4,1440,RSV2-001440,2026-04-21,SAN MARCOS,0
1857,5,1858,RSV2-001858,2026-04-20,JINOTEPE,0
1936,6,1937,RSV2-001937,2026-03-28,DIRIAMBA,1
1943,7,1944,RSV2-001944,2026-04-25,SAN MARCOS,0
1989,8,1990,RSV2-001990,2026-01-06,DOLORES,1
2021,9,2022,RSV2-002022,2026-01-17,JINOTEPE,1
2067,10,2068,RSV2-002068,2026-06-02,JINOTEPE,0



CONTROLES DE AUDITORÍA: SUPERADOS
Guardado: auditoria_general_20261024T212136Z.csv
Guardado: resumen_auditoria_20261024T212136Z.csv
Guardado: tipos_hallazgo_20261024T212136Z.csv
Guardado: estado_todos_los_registros_20261024T212136Z.csv
Guardado: identificadores_repetidos_20261024T212136Z.csv
Guardado: filas_duplicadas_20261024T212136Z.csv
Guardado: POL-02_detalle_20261024T212136Z.csv | Incidencias: 920
Guardado: POL-03_detalle_20261024T212136Z.csv | Incidencias: 273
Guardado: POL-04_detalle_20261024T212136Z.csv | Incidencias: 254
Guardado: POL-05_detalle_20261024T212136Z.csv | Incidencias: 3

PASO 3 FINALIZADO
Carpeta: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\optativo_IV\auditoria
SQL Server permanece sin modificaciones.



## PASO 4. Pruebas funcionales de las políticas de Gobierno de Datos

### Objetivo
Verificar que las políticas programadas detecten correctamente datos inválidos y acepten datos que cumplen las reglas.

### Metodología
Se ejecutan ocho casos de prueba: uno válido y uno inválido para cada política POL-02, POL-03, POL-04 y POL-05.

### Evidencias
- Datos utilizados en las pruebas.
- Resultado esperado.
- Resultado obtenido.
- Incumplimientos detectados.
- Estado de cada prueba.

Los datos de prueba se generan en memoria y no se insertan en SQL Server.


In [4]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO IV - GOBIERNO DE DATOS
# PASO 4. PRUEBAS CON VALORES VÁLIDOS E INVÁLIDOS
# ============================================================

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display


# ============================================================
# 1. PREPARAR CARPETA DE EVIDENCIAS
# ============================================================

carpeta_pruebas_gob = (
    proyecto
    / "resultados_jupyter"
    / "optativo_IV"
    / "pruebas"
)

carpeta_pruebas_gob.mkdir(
    parents=True,
    exist_ok=True
)

fecha_pruebas = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ"
)


# ============================================================
# 2. OBTENER UN REGISTRO BASE VÁLIDO
# ============================================================

# Elegimos un registro original que haya superado
# todas las políticas evaluadas en el PASO 3.

ids_validos = estado_registros_gob.loc[
    estado_registros_gob["Estado_auditoria"]
    == "SIN_HALLAZGOS",
    "id_raw"
]

if ids_validos.empty:
    raise ValueError(
        "No hay registros base sin hallazgos."
    )

id_base = ids_validos.iloc[0]

registro_base = (
    datos_gobierno.loc[
        datos_gobierno["id_raw"] == id_base
    ]
    .iloc[0]
    .copy()
)


# ============================================================
# 3. CREAR REGISTROS DE PRUEBA
# ============================================================

contador_id = 0

def crear_registro_prueba(etiqueta, cambios=None):

    global contador_id

    contador_id += 1

    registro = registro_base.copy()

    # Identificador técnico exclusivo para la prueba
    registro["id_raw"] = 10000000 + contador_id

    # Identificador de cita de prueba
    registro["id_cita"] = (
        f"PRUEBA-GOB-{etiqueta}"
    )

    if cambios:
        for campo, valor in cambios.items():
            registro[campo] = valor

    return registro


# ============================================================
# 4. DEFINIR LOS OCHO CASOS DE PRUEBA
# ============================================================

casos_prueba = [
    {
        "Politica": "POL-02",
        "Caso": "Registro con campos completos",
        "Tipo": "VALIDO",
        "Esperado": "ACEPTADO",
        "Datos": [
            crear_registro_prueba("P02-V")
        ]
    },
    {
        "Politica": "POL-02",
        "Caso": "Centro de salud vacío",
        "Tipo": "INVALIDO",
        "Esperado": "RECHAZADO",
        "Datos": [
            crear_registro_prueba(
                "P02-I",
                {"centro_salud": None}
            )
        ]
    },
    {
        "Politica": "POL-03",
        "Caso": "Categoría del catálogo",
        "Tipo": "VALIDO",
        "Esperado": "ACEPTADO",
        "Datos": [
            crear_registro_prueba("P03-V")
        ]
    },
    {
        "Politica": "POL-03",
        "Caso": "Centro fuera de catálogo",
        "Tipo": "INVALIDO",
        "Esperado": "RECHAZADO",
        "Datos": [
            crear_registro_prueba(
                "P03-I",
                {
                    "centro_salud":
                        "CENTRO NO REGISTRADO"
                }
            )
        ]
    },
    {
        "Politica": "POL-04",
        "Caso": "Identificadores diferentes",
        "Tipo": "VALIDO",
        "Esperado": "ACEPTADO",
        "Datos": [
            crear_registro_prueba("P04-V-A"),
            crear_registro_prueba("P04-V-B")
        ]
    },
    {
        "Politica": "POL-04",
        "Caso": "Identificador duplicado",
        "Tipo": "INVALIDO",
        "Esperado": "RECHAZADO",
        "Datos": [
            crear_registro_prueba("P04-I"),
            crear_registro_prueba("P04-I")
        ]
    },
    {
        "Politica": "POL-05",
        "Caso": "Días de espera válidos",
        "Tipo": "VALIDO",
        "Esperado": "ACEPTADO",
        "Datos": [
            crear_registro_prueba(
                "P05-V",
                {"espera_dias": 5}
            )
        ]
    },
    {
        "Politica": "POL-05",
        "Caso": "Días de espera negativos",
        "Tipo": "INVALIDO",
        "Esperado": "RECHAZADO",
        "Datos": [
            crear_registro_prueba(
                "P05-I",
                {"espera_dias": -5}
            )
        ]
    }
]


# ============================================================
# 5. EJECUTAR LAS PRUEBAS
# ============================================================

resultados_pruebas = []
detalles_incidencias = []
datos_utilizados = []

for numero, caso in enumerate(
    casos_prueba,
    start=1
):

    lote = pd.DataFrame(
        caso["Datos"]
    )

    # Guardar los datos exactos que fueron probados
    copia_evidencia = lote.copy()

    copia_evidencia["Numero_prueba"] = numero
    copia_evidencia["Politica_probada"] = (
        caso["Politica"]
    )
    copia_evidencia["Caso"] = caso["Caso"]
    copia_evidencia["Tipo_prueba"] = caso["Tipo"]

    datos_utilizados.append(copia_evidencia)

    # Utilizar la misma función del PASO 2
    hallazgos, resumen = (
        evaluar_politicas_gobierno(lote)
    )

    # Identificar los registros afectados
    afectados = int(
        hallazgos["id_raw"].nunique()
    )

    # Una prueba inválida debe generar errores
    # de su política correspondiente.
    politicas_detectadas = set(
        hallazgos["Politica"]
    )

    if hallazgos.empty:
        obtenido = "ACEPTADO"
    else:
        obtenido = "RECHAZADO"

    # Comprobar el comportamiento de la política
    if caso["Tipo"] == "VALIDO":

        correcto = (
            obtenido == "ACEPTADO"
            and len(hallazgos) == 0
        )

    else:

        correcto = (
            obtenido == "RECHAZADO"
            and politicas_detectadas
                == {caso["Politica"]}
            and afectados == len(lote)
        )

    resultado = (
        "SUPERADA"
        if correcto
        else "FALLIDA"
    )

    # Descripción del valor que causó la incidencia
    descripcion = "SIN INCUMPLIMIENTOS"

    if not hallazgos.empty:

        descripcion = " | ".join(
            f"{fila.Politica}: "
            f"{fila.Campo} = {fila.Valor_original}"
            for fila in hallazgos.itertuples()
        )

    resultados_pruebas.append({
        "Numero": numero,
        "Politica": caso["Politica"],
        "Caso": caso["Caso"],
        "Tipo": caso["Tipo"],
        "Registros_probados": len(lote),
        "Esperado": caso["Esperado"],
        "Obtenido": obtenido,
        "Registros_rechazados": afectados,
        "Incidencias": len(hallazgos),
        "Detalle": descripcion,
        "Resultado": resultado
    })

    if not hallazgos.empty:

        detalle_caso = hallazgos.copy()

        detalle_caso["Numero_prueba"] = numero
        detalle_caso["Caso"] = caso["Caso"]

        detalles_incidencias.append(
            detalle_caso
        )


# ============================================================
# 6. CONSTRUIR TABLAS DE EVIDENCIA
# ============================================================

tabla_pruebas_gob = pd.DataFrame(
    resultados_pruebas
)

datos_pruebas_gob = pd.concat(
    datos_utilizados,
    ignore_index=True
)

if detalles_incidencias:

    detalle_pruebas_gob = pd.concat(
        detalles_incidencias,
        ignore_index=True
    )

else:

    detalle_pruebas_gob = pd.DataFrame()


# ============================================================
# 7. MOSTRAR LOS RESULTADOS
# ============================================================

print("=" * 70)
print("OPTATIVO IV - GOBIERNO DE DATOS")
print("PASO 4. PRUEBAS FUNCIONALES")
print("=" * 70)

print("\n1. RESULTADOS DE LAS OCHO PRUEBAS")

display(
    tabla_pruebas_gob[
        [
            "Politica",
            "Caso",
            "Tipo",
            "Esperado",
            "Obtenido",
            "Resultado"
        ]
    ]
)

print("\n2. DETALLE DE CADA PRUEBA")

display(
    tabla_pruebas_gob[
        [
            "Numero",
            "Politica",
            "Registros_probados",
            "Registros_rechazados",
            "Incidencias",
            "Detalle"
        ]
    ]
)

print("\n3. EJEMPLOS DE DATOS UTILIZADOS")

display(
    datos_pruebas_gob[
        [
            "Numero_prueba",
            "Politica_probada",
            "Caso",
            "id_raw",
            "id_cita",
            "centro_salud",
            "espera_dias"
        ]
    ]
)

superadas = int(
    tabla_pruebas_gob["Resultado"]
    .eq("SUPERADA")
    .sum()
)

print("\nRESUMEN DE PRUEBAS")
print("Pruebas ejecutadas:", len(tabla_pruebas_gob))
print("Pruebas superadas:", superadas)
print(
    "Pruebas fallidas:",
    len(tabla_pruebas_gob) - superadas
)


# ============================================================
# 8. GUARDAR EVIDENCIAS
# ============================================================

archivos_pruebas = {
    "resultados_pruebas": tabla_pruebas_gob,
    "datos_utilizados": datos_pruebas_gob,
    "incidencias_detectadas": detalle_pruebas_gob
}

for nombre, tabla in archivos_pruebas.items():

    ruta = (
        carpeta_pruebas_gob
        / f"{nombre}_{fecha_pruebas}.csv"
    )

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", ruta.name)


# ============================================================
# 9. VERIFICAR EL RESULTADO GENERAL
# ============================================================

if superadas != 8:

    raise AssertionError(
        "No se superaron las ocho pruebas. "
        "Revisa la tabla de resultados."
    )

print("\nLAS OCHO PRUEBAS FUERON SUPERADAS")
print("SQL Server no fue modificado.")
print("Carpeta de evidencias:", carpeta_pruebas_gob)


OPTATIVO IV - GOBIERNO DE DATOS
PASO 4. PRUEBAS FUNCIONALES

1. RESULTADOS DE LAS OCHO PRUEBAS


,Politica,Caso,Tipo,Esperado,Obtenido,Resultado
0,POL-02,Registro con campos completos,VALIDO,ACEPTADO,ACEPTADO,SUPERADA
1,POL-02,Centro de salud vacío,INVALIDO,RECHAZADO,RECHAZADO,SUPERADA
2,POL-03,Categoría del catálogo,VALIDO,ACEPTADO,ACEPTADO,SUPERADA
3,POL-03,Centro fuera de catálogo,INVALIDO,RECHAZADO,RECHAZADO,SUPERADA
4,POL-04,Identificadores diferentes,VALIDO,ACEPTADO,ACEPTADO,SUPERADA
5,POL-04,Identificador duplicado,INVALIDO,RECHAZADO,RECHAZADO,SUPERADA
6,POL-05,Días de espera válidos,VALIDO,ACEPTADO,ACEPTADO,SUPERADA
7,POL-05,Días de espera negativos,INVALIDO,RECHAZADO,RECHAZADO,SUPERADA



2. DETALLE DE CADA PRUEBA


,Numero,Politica,Registros_probados,Registros_rechazados,Incidencias,Detalle
0,1,POL-02,1,0,0,SIN INCUMPLIMIENTOS
1,2,POL-02,1,1,1,POL-02: centro_salud = <NULO>
2,3,POL-03,1,0,0,SIN INCUMPLIMIENTOS
3,4,POL-03,1,1,1,POL-03: centro_salud = CENTRO NO REGISTRADO
4,5,POL-04,2,0,0,SIN INCUMPLIMIENTOS
5,6,POL-04,2,2,4,POL-04: id_cita = PRUEBA-GOB-P04-I | POL-04: i...
6,7,POL-05,1,0,0,SIN INCUMPLIMIENTOS
7,8,POL-05,1,1,1,POL-05: espera_dias = -5



3. EJEMPLOS DE DATOS UTILIZADOS


,Numero_prueba,Politica_probada,Caso,id_raw,id_cita,centro_salud,espera_dias
0,1,POL-02,Registro con campos completos,10000001,PRUEBA-GOB-P02-V,DIRIAMBA,33.0
1,2,POL-02,Centro de salud vacío,10000002,PRUEBA-GOB-P02-I,None,33.0
2,3,POL-03,Categoría del catálogo,10000003,PRUEBA-GOB-P03-V,DIRIAMBA,33.0
3,4,POL-03,Centro fuera de catálogo,10000004,PRUEBA-GOB-P03-I,CENTRO NO REGISTRADO,33.0
4,5,POL-04,Identificadores diferentes,10000005,PRUEBA-GOB-P04-V-A,DIRIAMBA,33.0
5,5,POL-04,Identificadores diferentes,10000006,PRUEBA-GOB-P04-V-B,DIRIAMBA,33.0
6,6,POL-04,Identificador duplicado,10000007,PRUEBA-GOB-P04-I,DIRIAMBA,33.0
7,6,POL-04,Identificador duplicado,10000008,PRUEBA-GOB-P04-I,DIRIAMBA,33.0
8,7,POL-05,Días de espera válidos,10000009,PRUEBA-GOB-P05-V,DIRIAMBA,5.0
9,8,POL-05,Días de espera negativos,10000010,PRUEBA-GOB-P05-I,DIRIAMBA,-5.0



RESUMEN DE PRUEBAS
Pruebas ejecutadas: 8
Pruebas superadas: 8
Pruebas fallidas: 0
Guardado: resultados_pruebas_20261024T212520Z.csv
Guardado: datos_utilizados_20261024T212520Z.csv
Guardado: incidencias_detectadas_20261024T212520Z.csv

LAS OCHO PRUEBAS FUERON SUPERADAS
SQL Server no fue modificado.
Carpeta de evidencias: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\optativo_IV\pruebas



## PASO 5. Indicadores de calidad establecidos en el II corte

### Objetivo
Programar y verificar los tres KPI de calidad incluidos en el documento definitivo del II corte.

### Indicadores
- IND-01: Tasa de integridad de registros de citas. Meta >98%.
- IND-02: Porcentaje de registros con categorías no normalizadas. Meta <1%.
- IND-04: Porcentaje de filas duplicadas. Meta 0%.

### Metodología
Se calculan los mismos indicadores sobre los registros originales y el conjunto depurado.

El proceso conserva la evidencia de los registros afectados, las fórmulas, los resultados y la cantidad de datos disponibles después del tratamiento.

Los catálogos utilizados son provisionales. Los resultados posteriores corresponden al conjunto apto y no implican que todos los registros originales hayan sido corregidos.


In [5]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO IV - GOBIERNO DE DATOS
# PASO 5. TRES KPI OFICIALES DEL II CORTE
# ============================================================

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display


# ============================================================
# 1. PREPARAR LAS FUENTES
# ============================================================

datos_origen_kpi = datos_gobierno.copy()

# Recuperar el conjunto depurado generado por Calidad
carpeta_origen_depurado = (
    proyecto
    / "resultados_jupyter"
    / "optativo_III"
    / "limpieza"
)

archivos_depurados = list(
    carpeta_origen_depurado.glob("dataset_apto_*.csv")
)

if not archivos_depurados:
    raise FileNotFoundError(
        "No se encontró el dataset apto del Optativo III."
    )

archivo_depurado = max(
    archivos_depurados,
    key=lambda archivo: archivo.stat().st_mtime
)

datos_depurados_kpi = pd.read_csv(
    archivo_depurado,
    dtype={
        "id_cita": "string",
        "no_asistio": "string"
    },
    low_memory=False
)

# Carpeta exclusiva de KPI del Optativo IV
carpeta_kpi_gob = (
    proyecto
    / "resultados_jupyter"
    / "optativo_IV"
    / "kpi_calidad"
)

carpeta_kpi_gob.mkdir(
    parents=True,
    exist_ok=True
)

fecha_kpi_gob = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ"
)


# ============================================================
# 2. VERIFICAR LAS FUENTES
# ============================================================

columnas_necesarias = set(
    campos_obligatorios_gob
    + columnas_cita_gob
    + ["id_raw"]
)

for nombre, datos in [
    ("ORIGINAL", datos_origen_kpi),
    ("DEPURADO", datos_depurados_kpi)
]:

    faltantes = columnas_necesarias - set(datos.columns)

    if faltantes:
        raise ValueError(
            f"Faltan columnas en {nombre}: "
            f"{sorted(faltantes)}"
        )

    if not datos["id_raw"].is_unique:
        raise ValueError(
            f"Existen id_raw repetidos en {nombre}."
        )

if not set(datos_depurados_kpi["id_raw"]).issubset(
    set(datos_origen_kpi["id_raw"])
):
    raise ValueError(
        "El dataset depurado contiene registros "
        "que no aparecen en el origen."
    )


# ============================================================
# 3. FUNCIÓN REUTILIZABLE DE LOS TRES KPI
# ============================================================

def evaluar_kpi_gobierno(datos):

    n = len(datos)

    if n == 0:
        raise ValueError("No hay registros para evaluar.")

    # --------------------------------------------------------
    # IND-01. INTEGRIDAD
    # --------------------------------------------------------

    faltantes = pd.DataFrame({
        campo: detectar_vacios_gob(datos[campo])
        for campo in campos_obligatorios_gob
    })

    registros_completos = int(
        (~faltantes.any(axis=1)).sum()
    )

    integridad = (
        registros_completos / n * 100
    )

    # --------------------------------------------------------
    # IND-02. CATEGORÍAS NO NORMALIZADAS
    # --------------------------------------------------------

    categorizados = pd.Series(
        False,
        index=datos.index
    )

    no_normalizados = pd.Series(
        False,
        index=datos.index
    )

    detalle_categorias = []

    for campo, permitidos in catalogos_gob.items():

        serie = datos[campo].astype("string")

        tiene_valor = (
            ~detectar_vacios_gob(serie)
        ).fillna(False)

        equivalencias = {
            normalizar_texto_gob(valor): valor
            for valor in permitidos
        }

        claves = serie.map(normalizar_texto_gob)

        valores_esperados = claves.map(equivalencias)

        fuera_catalogo = (
            tiene_valor
            & ~claves.isin(equivalencias)
        ).fillna(False)

        formato_diferente = (
            tiene_valor
            & claves.isin(equivalencias)
            & serie.ne(
                valores_esperados.astype("string")
            ).fillna(False)
        )

        incumple = (
            fuera_catalogo | formato_diferente
        ).fillna(False)

        categorizados |= tiene_valor
        no_normalizados |= incumple

        # Registrar cada diferencia para auditoría
        for indice in datos.index[incumple]:

            fila = datos.loc[indice]

            detalle_categorias.append({
                "id_raw": fila["id_raw"],
                "id_cita": fila["id_cita"],
                "Campo": campo,
                "Valor_original": fila[campo],
                "Valor_esperado": (
                    "REQUIERE REVISIÓN"
                    if fuera_catalogo.loc[indice]
                    else valores_esperados.loc[indice]
                ),
                "Tipo_hallazgo": (
                    "FUERA_DE_CATALOGO"
                    if fuera_catalogo.loc[indice]
                    else "FORMATO_NO_ESTANDAR"
                )
            })

    cantidad_categorizados = int(
        categorizados.sum()
    )

    cantidad_no_normalizados = int(
        no_normalizados.sum()
    )

    if cantidad_categorizados == 0:
        raise ValueError(
            "No hay registros categorizados."
        )

    porcentaje_no_normalizados = (
        cantidad_no_normalizados
        / cantidad_categorizados
        * 100
    )

    # --------------------------------------------------------
    # IND-04. FILAS DUPLICADAS
    # --------------------------------------------------------

    duplicados = datos.duplicated(
        subset=columnas_cita_gob,
        keep="first"
    )

    cantidad_duplicados = int(
        duplicados.sum()
    )

    porcentaje_duplicados = (
        cantidad_duplicados / n * 100
    )

    resultados = {
        "Registros": n,
        "Completos": registros_completos,
        "Categorizados": cantidad_categorizados,
        "No_normalizados": cantidad_no_normalizados,
        "Filas_duplicadas": cantidad_duplicados,
        "IND-01": integridad,
        "IND-02": porcentaje_no_normalizados,
        "IND-04": porcentaje_duplicados
    }

    detalle = pd.DataFrame(
        detalle_categorias,
        columns=[
            "id_raw",
            "id_cita",
            "Campo",
            "Valor_original",
            "Valor_esperado",
            "Tipo_hallazgo"
        ]
    )

    return resultados, detalle


# ============================================================
# 4. CALCULAR LOS KPI ANTES Y DESPUÉS
# ============================================================

kpi_antes_gob, categorias_originales_gob = (
    evaluar_kpi_gobierno(datos_origen_kpi)
)

kpi_despues_gob, categorias_depuradas_gob = (
    evaluar_kpi_gobierno(datos_depurados_kpi)
)


# ============================================================
# 5. TABLA DE RESULTADOS OFICIALES
# ============================================================

comparativo_gobierno = pd.DataFrame([
    {
        "Codigo": "IND-01",
        "KPI": "Tasa de integridad de registros de citas",
        "Antes": round(kpi_antes_gob["IND-01"], 2),
        "Despues": round(kpi_despues_gob["IND-01"], 2),
        "Meta": ">98%",
        "Estado": (
            "CUMPLE"
            if kpi_despues_gob["IND-01"] > 98
            else "NO CUMPLE"
        )
    },
    {
        "Codigo": "IND-02",
        "KPI": "Porcentaje de registros con categorías no normalizadas",
        "Antes": round(kpi_antes_gob["IND-02"], 2),
        "Despues": round(kpi_despues_gob["IND-02"], 2),
        "Meta": "<1%",
        "Estado": (
            "CUMPLE"
            if kpi_despues_gob["IND-02"] < 1
            else "NO CUMPLE"
        )
    },
    {
        "Codigo": "IND-04",
        "KPI": "Porcentaje de filas duplicadas",
        "Antes": round(kpi_antes_gob["IND-04"], 2),
        "Despues": round(kpi_despues_gob["IND-04"], 2),
        "Meta": "0%",
        "Estado": (
            "CUMPLE"
            if kpi_despues_gob["Filas_duplicadas"] == 0
            else "NO CUMPLE"
        )
    }
])


# ============================================================
# 6. RESUMEN DE CANTIDADES
# ============================================================

resumen_cantidades_gob = pd.DataFrame([
    {
        "Control": "Registros evaluados",
        "Original": kpi_antes_gob["Registros"],
        "Depurado": kpi_despues_gob["Registros"]
    },
    {
        "Control": "Registros completos",
        "Original": kpi_antes_gob["Completos"],
        "Depurado": kpi_despues_gob["Completos"]
    },
    {
        "Control": "Registros categorizados",
        "Original": kpi_antes_gob["Categorizados"],
        "Depurado": kpi_despues_gob["Categorizados"]
    },
    {
        "Control": "Registros con categorías no normalizadas",
        "Original": kpi_antes_gob["No_normalizados"],
        "Depurado": kpi_despues_gob["No_normalizados"]
    },
    {
        "Control": "Filas duplicadas adicionales",
        "Original": kpi_antes_gob["Filas_duplicadas"],
        "Depurado": kpi_despues_gob["Filas_duplicadas"]
    }
])

cobertura_gob = (
    len(datos_depurados_kpi)
    / len(datos_origen_kpi)
    * 100
)


# ============================================================
# 7. VERIFICACIONES
# ============================================================

assert set(comparativo_gobierno["Codigo"]) == {
    "IND-01", "IND-02", "IND-04"
}, "Los KPI no coinciden con el II corte."

assert (
    categorias_originales_gob["id_raw"].nunique()
    == kpi_antes_gob["No_normalizados"]
), "No coincide el IND-02 con su auditoría."

assert (
    kpi_antes_gob["No_normalizados"] == 273
), "Revisar diferencias respecto a POL-03."


# ============================================================
# 8. MOSTRAR RESULTADOS
# ============================================================

print("=" * 70)
print("OPTATIVO IV - GOBIERNO DE DATOS")
print("PASO 5. KPI DE CALIDAD DEL II CORTE")
print("=" * 70)

print("\nFUENTES")
print(
    "Registros originales:",
    len(datos_origen_kpi)
)
print(
    "Registros depurados:",
    len(datos_depurados_kpi)
)
print(
    "Archivo depurado:",
    archivo_depurado.name
)

print("\n1. LOS TRES KPI OFICIALES")
display(comparativo_gobierno)

print("\n2. RESPALDO NUMÉRICO")
display(resumen_cantidades_gob)

print("\n3. EJEMPLOS DE REGISTROS NO NORMALIZADOS")
display(categorias_originales_gob.head(10))

print(
    "\nCobertura del conjunto depurado:",
    f"{cobertura_gob:.2f}%"
)


# ============================================================
# 9. GUARDAR EVIDENCIAS
# ============================================================

archivos_kpi_gob = {
    "comparativo_kpi": comparativo_gobierno,
    "cantidades_kpi": resumen_cantidades_gob,
    "auditoria_ind02": categorias_originales_gob,
    "auditoria_ind02_depurado": categorias_depuradas_gob
}

for nombre, tabla in archivos_kpi_gob.items():

    ruta = (
        carpeta_kpi_gob
        / f"{nombre}_{fecha_kpi_gob}.csv"
    )

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", ruta.name)

print("\nPASO 5 FINALIZADO")
print("No se modificaron los registros de SQL Server.")


OPTATIVO IV - GOBIERNO DE DATOS
PASO 5. KPI DE CALIDAD DEL II CORTE

FUENTES
Registros originales: 10080
Registros depurados: 9076
Archivo depurado: dataset_apto_20261024T210524Z.csv

1. LOS TRES KPI OFICIALES


,Codigo,KPI,Antes,Despues,Meta,Estado
0,IND-01,Tasa de integridad de registros de citas,90.87,100.0,>98%,CUMPLE
1,IND-02,Porcentaje de registros con categorías no norm...,2.71,0.0,<1%,CUMPLE
2,IND-04,Porcentaje de filas duplicadas,0.47,0.0,0%,CUMPLE



2. RESPALDO NUMÉRICO


,Control,Original,Depurado
0,Registros evaluados,10080,9076
1,Registros completos,9160,9076
2,Registros categorizados,10080,9076
3,Registros con categorías no normalizadas,273,0
4,Filas duplicadas adicionales,47,0



3. EJEMPLOS DE REGISTROS NO NORMALIZADOS


,id_raw,id_cita,Campo,Valor_original,Valor_esperado,Tipo_hallazgo
0,102,RSV2-000102,centro_salud,jinotepe,JINOTEPE,FORMATO_NO_ESTANDAR
1,258,RSV2-000258,centro_salud,jinotepe,JINOTEPE,FORMATO_NO_ESTANDAR
2,379,RSV2-000379,centro_salud,dolores,DOLORES,FORMATO_NO_ESTANDAR
3,538,RSV2-000538,centro_salud,dolores,DOLORES,FORMATO_NO_ESTANDAR
4,739,RSV2-000739,centro_salud,dolores,DOLORES,FORMATO_NO_ESTANDAR
5,846,RSV2-000846,centro_salud,jinotepe,JINOTEPE,FORMATO_NO_ESTANDAR
6,945,RSV2-000945,centro_salud,san marcos,SAN MARCOS,FORMATO_NO_ESTANDAR
7,1174,RSV2-001174,centro_salud,diriamba,DIRIAMBA,FORMATO_NO_ESTANDAR
8,1184,RSV2-001184,centro_salud,jinotepe,JINOTEPE,FORMATO_NO_ESTANDAR
9,1254,RSV2-001254,centro_salud,diriamba,DIRIAMBA,FORMATO_NO_ESTANDAR



Cobertura del conjunto depurado: 90.04%
Guardado: comparativo_kpi_20261024T212803Z.csv
Guardado: cantidades_kpi_20261024T212803Z.csv
Guardado: auditoria_ind02_20261024T212803Z.csv
Guardado: auditoria_ind02_depurado_20261024T212803Z.csv

PASO 5 FINALIZADO
No se modificaron los registros de SQL Server.



## PASO 6. Trazabilidad de datos y decisiones de gobierno

### Objetivo
Relacionar los registros originales con los hallazgos de auditoría, las políticas aplicadas y las decisiones de tratamiento.

### Evidencias
- Identificación del registro original.
- Procedencia y lote de carga.
- Políticas incumplidas e incidencias detectadas.
- Normalizaciones registradas.
- Estado posterior al tratamiento.
- Correspondencia con el conjunto apto.

### Alcance
Se verifica el recorrido desde dbo.raw_cita hasta la auditoría y el conjunto apto generado en Gestión de la Calidad.

Los registros originales permanecen conservados en SQL Server. Los registros en cuarentena continúan pendientes de revisión.


In [6]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO IV - GOBIERNO DE DATOS
# PASO 6. TRAZABILIDAD Y SEGUIMIENTO DE REGISTROS
# ============================================================

import pandas as pd

from datetime import datetime, timezone
from IPython.display import display


# ============================================================
# 1. VERIFICAR LAS VARIABLES DE PASOS ANTERIORES
# ============================================================

variables_requeridas = [
    "proyecto",
    "datos_gobierno",
    "auditoria",
    "datos_depurados_kpi",
    "archivo_depurado"
]

faltantes = [
    nombre
    for nombre in variables_requeridas
    if nombre not in globals()
]

if faltantes:
    raise RuntimeError(
        "Ejecuta los pasos anteriores. Faltan: "
        + ", ".join(faltantes)
    )


# ============================================================
# 2. PREPARAR CARPETA DE TRAZABILIDAD
# ============================================================

carpeta_trazabilidad = (
    proyecto
    / "resultados_jupyter"
    / "optativo_IV"
    / "trazabilidad"
)

carpeta_trazabilidad.mkdir(
    parents=True,
    exist_ok=True
)

fecha_trazabilidad = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")


# ============================================================
# 3. RECUPERAR LAS EVIDENCIAS DE LIMPIEZA
# ============================================================

# Se utiliza la misma ejecución de limpieza
# que produjo el dataset apto del PASO 5.

marca_ejecucion = (
    archivo_depurado.stem.replace(
        "dataset_apto_", "", 1
    )
)

carpeta_limpieza = archivo_depurado.parent

ruta_bitacora = (
    carpeta_limpieza
    / f"bitacora_tratamientos_{marca_ejecucion}.csv"
)

ruta_normalizaciones = (
    carpeta_limpieza
    / f"detalle_normalizaciones_{marca_ejecucion}.csv"
)

for ruta in [ruta_bitacora, ruta_normalizaciones]:

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró la evidencia: {ruta.name}"
        )

bitacora_gob = pd.read_csv(
    ruta_bitacora,
    low_memory=False
)

normalizaciones_gob = pd.read_csv(
    ruta_normalizaciones,
    low_memory=False
)

origen = datos_gobierno.copy()
apto = datos_depurados_kpi.copy()
hallazgos = auditoria.copy()


# ============================================================
# 4. NORMALIZAR IDENTIFICADORES TÉCNICOS
# ============================================================

for tabla in [
    origen,
    apto,
    bitacora_gob,
    normalizaciones_gob,
    hallazgos
]:

    tabla["id_raw"] = pd.to_numeric(
        tabla["id_raw"],
        errors="raise"
    ).astype("int64")

for nombre, tabla in [
    ("ORIGEN", origen),
    ("CONJUNTO_APTO", apto),
    ("BITACORA", bitacora_gob)
]:

    if not tabla["id_raw"].is_unique:
        raise ValueError(
            f"Existen id_raw repetidos en {nombre}."
        )


# ============================================================
# 5. IDENTIFICAR LAS INCIDENCIAS POR REGISTRO
# ============================================================

incidencias_por_registro = (
    hallazgos
    .groupby("id_raw")
    .agg(
        Total_incidencias=("Politica", "size"),
        Politicas_incumplidas=(
            "Politica",
            lambda valores: ", ".join(
                sorted(set(valores))
            )
        )
    )
    .reset_index()
)


# ============================================================
# 6. IDENTIFICAR LAS NORMALIZACIONES POR REGISTRO
# ============================================================

normalizaciones_por_registro = (
    normalizaciones_gob
    .groupby("id_raw")
    .agg(
        Campos_normalizados=("Campo", "count"),
        Politica_normalizacion=(
            "Politica",
            lambda valores: ", ".join(
                sorted(set(valores))
            )
        )
    )
    .reset_index()
)


# ============================================================
# 7. CONSTRUIR LA TRAZABILIDAD INDIVIDUAL
# ============================================================

campos_procedencia = [
    "id_raw",
    "id_cita",
    "origen_dataset",
    "lote_carga",
    "fecha_ingesta"
]

trazabilidad_registros = origen[
    campos_procedencia
].copy()

# Incorporar los hallazgos de Gobierno
trazabilidad_registros = (
    trazabilidad_registros.merge(
        incidencias_por_registro,
        on="id_raw",
        how="left",
        validate="one_to_one"
    )
)

# Incorporar las decisiones de Calidad
columnas_bitacora = [
    "id_raw",
    "Motivos",
    "Campos_faltantes",
    "Estado_final"
]

trazabilidad_registros = (
    trazabilidad_registros.merge(
        bitacora_gob[columnas_bitacora],
        on="id_raw",
        how="left",
        validate="one_to_one"
    )
)

# Incorporar las normalizaciones
trazabilidad_registros = (
    trazabilidad_registros.merge(
        normalizaciones_por_registro,
        on="id_raw",
        how="left",
        validate="one_to_one"
    )
)

# Completar campos informativos
trazabilidad_registros["Total_incidencias"] = (
    trazabilidad_registros["Total_incidencias"]
    .fillna(0)
    .astype(int)
)

trazabilidad_registros["Campos_normalizados"] = (
    trazabilidad_registros["Campos_normalizados"]
    .fillna(0)
    .astype(int)
)

trazabilidad_registros["Politicas_incumplidas"] = (
    trazabilidad_registros["Politicas_incumplidas"]
    .fillna("NINGUNA")
)

trazabilidad_registros["Motivos"] = (
    trazabilidad_registros["Motivos"]
    .fillna("SIN MOTIVOS DE EXCLUSION")
)

trazabilidad_registros["Campos_faltantes"] = (
    trazabilidad_registros["Campos_faltantes"]
    .fillna("")
)

trazabilidad_registros["Condicion_auditoria"] = (
    trazabilidad_registros["Total_incidencias"]
    .apply(
        lambda cantidad: (
            "CON_HALLAZGOS"
            if cantidad > 0
            else "SIN_HALLAZGOS"
        )
    )
)

# Identificar si el registro aparece en la
# versión depurada seleccionada.

ids_aptos = set(apto["id_raw"])

trazabilidad_registros["En_dataset_apto"] = (
    trazabilidad_registros["id_raw"]
    .isin(ids_aptos)
)

trazabilidad_registros["Fuente_original"] = (
    "dbo.raw_cita"
)

trazabilidad_registros["Version_depurada"] = (
    archivo_depurado.name
)


# ============================================================
# 8. RELACIONAR LAS INCIDENCIAS CON SU DESTINO
# ============================================================

detalle_auditoria_trazable = hallazgos.merge(
    trazabilidad_registros[
        [
            "id_raw",
            "Estado_final",
            "En_dataset_apto",
            "Campos_normalizados",
            "Motivos"
        ]
    ],
    on="id_raw",
    how="left",
    validate="many_to_one"
)

# Relacionar cada cambio con el destino del registro
normalizaciones_trazables = normalizaciones_gob.merge(
    trazabilidad_registros[
        [
            "id_raw",
            "Estado_final",
            "En_dataset_apto"
        ]
    ],
    on="id_raw",
    how="left",
    validate="many_to_one"
)


# ============================================================
# 9. VERIFICAR LA CORRESPONDENCIA DE LOS REGISTROS
# ============================================================

# Todo registro original debe estar documentado.
assert (
    len(trazabilidad_registros) == len(origen)
), "Faltan registros en la trazabilidad."

assert (
    set(trazabilidad_registros["id_raw"])
    == set(origen["id_raw"])
), "Los identificadores no coinciden."

# La bitácora debe cubrir el origen completo.
assert (
    set(bitacora_gob["id_raw"])
    == set(origen["id_raw"])
), "La bitácora no cubre todo el origen."

# Todo registro apto debe existir en el origen.
assert ids_aptos.issubset(
    set(origen["id_raw"])
), "Existen registros aptos sin origen."

# El estado APTO debe coincidir con el archivo.
mascara_apto = (
    trazabilidad_registros["Estado_final"]
    == "APTO"
)

assert (
    mascara_apto
    == trazabilidad_registros["En_dataset_apto"]
).all(), (
    "La decisión de la bitácora no coincide "
    "con el dataset apto."
)

# Ninguna incidencia puede referirse
# a un registro inexistente.
assert set(hallazgos["id_raw"]).issubset(
    set(origen["id_raw"])
), "La auditoría contiene IDs desconocidos."

# Ninguna normalización puede referirse
# a un registro inexistente.
assert set(normalizaciones_gob["id_raw"]).issubset(
    set(origen["id_raw"])
), "Hay normalizaciones sin registro original."


# ============================================================
# 10. CONSTRUIR EL RESUMEN DE DESTINOS
# ============================================================

resumen_destinos_gob = (
    trazabilidad_registros
    .groupby("Estado_final")
    .agg(
        Registros=("id_raw", "size"),
        Con_hallazgos=(
            "Total_incidencias",
            lambda valores: int(
                (valores > 0).sum()
            )
        ),
        Registros_normalizados=(
            "Campos_normalizados",
            lambda valores: int(
                (valores > 0).sum()
            )
        )
    )
    .reset_index()
)

resumen_destinos_gob["Porcentaje"] = (
    resumen_destinos_gob["Registros"]
    / len(origen)
    * 100
).round(2)


# ============================================================
# 11. RESUMEN DE POLÍTICAS Y SEGUIMIENTO
# ============================================================

resumen_politicas_trazabilidad = (
    detalle_auditoria_trazable
    .groupby(["Politica", "Estado_final"])
    .agg(
        Incidencias=("id_raw", "size"),
        Registros=("id_raw", "nunique")
    )
    .reset_index()
)


# ============================================================
# 12. MOSTRAR RESULTADOS
# ============================================================

print("=" * 70)
print("OPTATIVO IV - GOBIERNO DE DATOS")
print("PASO 6. TRAZABILIDAD DE REGISTROS")
print("=" * 70)

print("\n1. VERIFICACIÓN DE FUENTES")

print("Registros originales:", len(origen))
print("Registros aptos:", len(apto))
print("Incidencias auditadas:", len(hallazgos))
print("Normalizaciones registradas:", len(normalizaciones_gob))

print("\n2. DESTINO DE LOS REGISTROS")
display(resumen_destinos_gob)

print("\n3. SEGUIMIENTO POR POLÍTICA")
display(resumen_politicas_trazabilidad)

print("\n4. EJEMPLOS DE TRAZABILIDAD")

columnas_muestra = [
    "id_raw",
    "id_cita",
    "Politicas_incumplidas",
    "Total_incidencias",
    "Campos_normalizados",
    "Estado_final",
    "En_dataset_apto"
]

display(
    trazabilidad_registros[
        columnas_muestra
    ].head(15)
)

print("\n5. EJEMPLOS DE NORMALIZACIONES")

display(
    normalizaciones_trazables[
        [
            "id_raw",
            "id_cita",
            "Campo",
            "Valor_antes",
            "Valor_despues",
            "Estado_final"
        ]
    ].head(10)
)

print("\nCONTROLES DE TRAZABILIDAD: SUPERADOS")


# ============================================================
# 13. GUARDAR EVIDENCIAS
# ============================================================

archivos_trazabilidad = {
    "trazabilidad_todos_registros":
        trazabilidad_registros,
    "incidencias_con_destino":
        detalle_auditoria_trazable,
    "normalizaciones_con_destino":
        normalizaciones_trazables,
    "resumen_destinos":
        resumen_destinos_gob,
    "seguimiento_por_politica":
        resumen_politicas_trazabilidad
}

for nombre, tabla in archivos_trazabilidad.items():

    ruta = (
        carpeta_trazabilidad
        / f"{nombre}_{fecha_trazabilidad}.csv"
    )

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", ruta.name)


# ============================================================
# 14. CIERRE DEL PASO
# ============================================================

print("\nPASO 6 FINALIZADO")

print("Carpeta:", carpeta_trazabilidad)

print(
    "Todos los registros originales tienen "
    "un estado documentado."
)

print(
    "No se modificaron registros de SQL Server."
)


OPTATIVO IV - GOBIERNO DE DATOS
PASO 6. TRAZABILIDAD DE REGISTROS

1. VERIFICACIÓN DE FUENTES
Registros originales: 10080
Registros aptos: 9076
Incidencias auditadas: 1450
Normalizaciones registradas: 273

2. DESTINO DE LOS REGISTROS


,Estado_final,Registros,Con_hallazgos,Registros_normalizados,Porcentaje
0,APTO,9076,320,240,90.04
1,EN_CUARENTENA_PARA_REVISION,924,923,0,9.17
2,EXCLUIDO_COPIA_CONFIRMADA,80,80,33,0.79



3. SEGUIMIENTO POR POLÍTICA


,Politica,Estado_final,Incidencias,Registros
0,POL-02,EN_CUARENTENA_PARA_REVISION,920,920
1,POL-03,APTO,240,240
2,POL-03,EXCLUIDO_COPIA_CONFIRMADA,33,33
3,POL-04,APTO,127,80
4,POL-04,EXCLUIDO_COPIA_CONFIRMADA,127,80
5,POL-05,EN_CUARENTENA_PARA_REVISION,3,3



4. EJEMPLOS DE TRAZABILIDAD


,id_raw,id_cita,Politicas_incumplidas,Total_incidencias,Campos_normalizados,Estado_final,En_dataset_apto
0,1,RSV2-000001,NINGUNA,0,0,APTO,True
1,2,RSV2-000002,NINGUNA,0,0,APTO,True
2,3,RSV2-000003,POL-02,1,0,EN_CUARENTENA_PARA_REVISION,False
3,4,RSV2-000004,NINGUNA,0,0,APTO,True
4,5,RSV2-000005,POL-02,1,0,EN_CUARENTENA_PARA_REVISION,False
5,6,RSV2-000006,NINGUNA,0,0,APTO,True
6,7,RSV2-000007,NINGUNA,0,0,APTO,True
7,8,RSV2-000008,NINGUNA,0,0,APTO,True
8,9,RSV2-000009,NINGUNA,0,0,APTO,True
9,10,RSV2-000010,NINGUNA,0,0,APTO,True



5. EJEMPLOS DE NORMALIZACIONES


,id_raw,id_cita,Campo,Valor_antes,Valor_despues,Estado_final
0,102,RSV2-000102,centro_salud,jinotepe,JINOTEPE,APTO
1,258,RSV2-000258,centro_salud,jinotepe,JINOTEPE,APTO
2,379,RSV2-000379,centro_salud,dolores,DOLORES,APTO
3,538,RSV2-000538,centro_salud,dolores,DOLORES,APTO
4,739,RSV2-000739,centro_salud,dolores,DOLORES,APTO
5,846,RSV2-000846,centro_salud,jinotepe,JINOTEPE,APTO
6,945,RSV2-000945,centro_salud,san marcos,SAN MARCOS,APTO
7,1174,RSV2-001174,centro_salud,diriamba,DIRIAMBA,APTO
8,1184,RSV2-001184,centro_salud,jinotepe,JINOTEPE,APTO
9,1254,RSV2-001254,centro_salud,diriamba,DIRIAMBA,APTO



CONTROLES DE TRAZABILIDAD: SUPERADOS
Guardado: trazabilidad_todos_registros_20261024T213108Z.csv
Guardado: incidencias_con_destino_20261024T213108Z.csv
Guardado: normalizaciones_con_destino_20261024T213108Z.csv
Guardado: resumen_destinos_20261024T213108Z.csv
Guardado: seguimiento_por_politica_20261024T213108Z.csv

PASO 6 FINALIZADO
Carpeta: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\optativo_IV\trazabilidad
Todos los registros originales tienen un estado documentado.
No se modificaron registros de SQL Server.



## PASO 7. Consolidación final del Gobierno de Datos

### Objetivo
Consolidar las evidencias de implementación, auditoría, pruebas, KPI y trazabilidad de las políticas establecidas en el Marco de Gobierno de Datos V2.

### Evidencias
Se presentan las políticas programadas, los incumplimientos detectados, las pruebas válidas e inválidas, los indicadores de calidad y el destino documentado de cada registro.

### Consideraciones
- POL-01 no se puede evaluar por falta de fecha de cierre.
- Los catálogos de POL-03 son provisionales.
- Los registros en cuarentena requieren revisión.
- Los KPI posteriores corresponden al conjunto apto.
- La base de datos original permanece sin modificaciones.


In [7]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO IV - GOBIERNO DE DATOS
# PASO 7. CONSOLIDACIÓN FINAL DE EVIDENCIAS
# ============================================================

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display


# ============================================================
# 1. VERIFICAR LOS RESULTADOS ANTERIORES
# ============================================================

variables_requeridas = [
    "proyecto",
    "datos_gobierno",
    "datos_depurados_kpi",
    "matriz_politicas",
    "auditoria",
    "resumen_gobierno",
    "tabla_pruebas_gob",
    "comparativo_gobierno",
    "trazabilidad_registros",
    "detalle_auditoria_trazable",
    "normalizaciones_trazables",
    "bitacora_gob",
    "archivo_depurado"
]

faltantes = [
    nombre
    for nombre in variables_requeridas
    if nombre not in globals()
]

if faltantes:
    raise RuntimeError(
        "Ejecuta los pasos anteriores. Faltan: "
        + ", ".join(faltantes)
    )


# ============================================================
# 2. CREAR CARPETA DEL RESUMEN FINAL
# ============================================================

carpeta_final_gob = (
    proyecto
    / "resultados_jupyter"
    / "optativo_IV"
    / "resumen_final"
)

carpeta_final_gob.mkdir(
    parents=True,
    exist_ok=True
)

fecha_final_gob = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")


# ============================================================
# 3. RESUMIR LA FUENTE Y LOS DESTINOS
# ============================================================

total_origen = len(datos_gobierno)
total_apto = len(datos_depurados_kpi)

estados = (
    trazabilidad_registros["Estado_final"]
    .value_counts()
)

total_cuarentena = int(
    estados.get("EN_CUARENTENA_PARA_REVISION", 0)
)

total_copias = int(
    estados.get("EXCLUIDO_COPIA_CONFIRMADA", 0)
)

cobertura = (
    total_apto / total_origen * 100
)

total_incidencias = len(auditoria)

registros_con_hallazgos = int(
    auditoria["id_raw"].nunique()
)

registros_sin_hallazgos = (
    total_origen - registros_con_hallazgos
)

total_normalizaciones = len(
    normalizaciones_trazables
)


# ============================================================
# 4. VERIFICAR TRAZABILIDAD GENERAL
# ============================================================

assert (
    total_origen
    == total_apto + total_cuarentena + total_copias
), "No coincide el balance de registros."

assert (
    len(trazabilidad_registros) == total_origen
), "Faltan registros en la trazabilidad."

assert (
    trazabilidad_registros["id_raw"].is_unique
), "Hay registros repetidos en la trazabilidad."

assert (
    set(trazabilidad_registros["id_raw"])
    == set(datos_gobierno["id_raw"])
), "No coinciden los identificadores originales."

ids_aptos = set(
    datos_depurados_kpi["id_raw"]
)

ids_aptos_bitacora = set(
    trazabilidad_registros.loc[
        trazabilidad_registros["Estado_final"] == "APTO",
        "id_raw"
    ]
)

assert (
    ids_aptos == ids_aptos_bitacora
), "El dataset apto no coincide con las decisiones."


# ============================================================
# 5. IDENTIFICAR CASOS ESPECIALES
# ============================================================

# Registros no aptos que NO presentan
# incumplimientos de las cuatro políticas.
# Pueden corresponder a revisiones adicionales del EDA.

excepciones = trazabilidad_registros.loc[
    (
        trazabilidad_registros["Estado_final"] != "APTO"
    )
    &
    (
        trazabilidad_registros["Total_incidencias"] == 0
    )
].copy()

# Registros aptos que sí tuvieron hallazgos
# antes del tratamiento.

aptos_con_hallazgos = trazabilidad_registros.loc[
    (
        trazabilidad_registros["Estado_final"] == "APTO"
    )
    &
    (
        trazabilidad_registros["Total_incidencias"] > 0
    )
].copy()


# ============================================================
# 6. MATRIZ FINAL DE POLÍTICAS
# ============================================================

politicas_finales = matriz_politicas.copy()

politicas_finales["Implementacion_tecnica"] = (
    politicas_finales["Codigo"].map({
        "POL-01": "NO EVALUABLE CON ESTOS DATOS",
        "POL-02": "PROGRAMADA",
        "POL-03": "PROGRAMADA",
        "POL-04": "PROGRAMADA",
        "POL-05": "PROGRAMADA"
    })
)

politicas_finales["Prueba_funcional"] = (
    politicas_finales["Codigo"].map({
        "POL-01": "NO EJECUTADA",
        "POL-02": "EJECUTADA",
        "POL-03": "EJECUTADA",
        "POL-04": "EJECUTADA",
        "POL-05": "EJECUTADA"
    })
)

# Estado de la matriz del PASO 1:
# "POR EVALUAR" corresponde al momento inicial.
# Estas columnas muestran la ejecución posterior.


# ============================================================
# 7. RESUMEN FINAL DE LA AUDITORÍA
# ============================================================

auditoria_final_gob = resumen_gobierno.copy()

assert (
    auditoria_final_gob["Incidencias"].sum()
    == total_incidencias
), "No coinciden las incidencias de auditoría."

seguimiento_final = (
    detalle_auditoria_trazable
    .groupby(
        ["Politica", "Estado_final"]
    )
    .agg(
        Incidencias=("id_raw", "size"),
        Registros=("id_raw", "nunique")
    )
    .reset_index()
)


# ============================================================
# 8. RESUMEN DE PRUEBAS FUNCIONALES
# ============================================================

pruebas_finales_gob = tabla_pruebas_gob.copy()

pruebas_superadas = int(
    pruebas_finales_gob["Resultado"]
    .eq("SUPERADA")
    .sum()
)

total_pruebas = len(pruebas_finales_gob)

assert total_pruebas == 8, (
    "Se esperaban ocho pruebas funcionales."
)

assert pruebas_superadas == total_pruebas, (
    "Existen pruebas funcionales fallidas."
)

# Comprobar que cada política tenga
# una prueba válida y otra inválida.

for politica in [
    "POL-02", "POL-03", "POL-04", "POL-05"
]:

    pruebas_politica = pruebas_finales_gob.loc[
        pruebas_finales_gob["Politica"] == politica
    ]

    assert set(pruebas_politica["Tipo"]) == {
        "VALIDO", "INVALIDO"
    }, f"Faltan casos válidos o inválidos en {politica}"


# ============================================================
# 9. VERIFICAR LOS TRES KPI DEL II CORTE
# ============================================================

kpi_finales_gob = comparativo_gobierno.copy()

codigos_esperados = {
    "IND-01",
    "IND-02",
    "IND-04"
}

assert (
    set(kpi_finales_gob["Codigo"])
    == codigos_esperados
), "Los KPI no coinciden con el II corte."

kpi_cumplidos = int(
    kpi_finales_gob["Estado"]
    .eq("CUMPLE")
    .sum()
)


# ============================================================
# 10. CREAR RESUMEN GENERAL
# ============================================================

resumen_general_gob = pd.DataFrame([
    {
        "Concepto": "Registros originales",
        "Cantidad": total_origen
    },
    {
        "Concepto": "Incidencias auditadas",
        "Cantidad": total_incidencias
    },
    {
        "Concepto": "Registros con hallazgos",
        "Cantidad": registros_con_hallazgos
    },
    {
        "Concepto": "Registros sin hallazgos",
        "Cantidad": registros_sin_hallazgos
    },
    {
        "Concepto": "Registros aptos",
        "Cantidad": total_apto
    },
    {
        "Concepto": "Registros en cuarentena",
        "Cantidad": total_cuarentena
    },
    {
        "Concepto": "Copias excluidas",
        "Cantidad": total_copias
    },
    {
        "Concepto": "Celdas normalizadas",
        "Cantidad": total_normalizaciones
    },
    {
        "Concepto": "Aptos con hallazgos iniciales",
        "Cantidad": len(aptos_con_hallazgos)
    },
    {
        "Concepto": "No aptos sin hallazgos de POL-02 a POL-05",
        "Cantidad": len(excepciones)
    },
    {
        "Concepto": "Pruebas superadas",
        "Cantidad": pruebas_superadas
    },
    {
        "Concepto": "KPI cumplidos en conjunto apto",
        "Cantidad": kpi_cumplidos
    }
])


# ============================================================
# 11. MOSTRAR RESULTADOS CONSOLIDADOS
# ============================================================

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")
print("OPTATIVO IV - RESUMEN FINAL DE GOBIERNO DE DATOS")
print("=" * 70)

print("\n1. MATRIZ DE POLÍTICAS")
display(
    politicas_finales[
        [
            "Codigo",
            "Politica",
            "Implementacion_tecnica",
            "Prueba_funcional"
        ]
    ]
)

print("\n2. AUDITORÍA POR POLÍTICA")
display(auditoria_final_gob)

print("\n3. SEGUIMIENTO DE HALLAZGOS")
display(seguimiento_final)

print("\n4. PRUEBAS FUNCIONALES")
display(
    pruebas_finales_gob[
        [
            "Politica",
            "Caso",
            "Esperado",
            "Obtenido",
            "Resultado"
        ]
    ]
)

print("\n5. KPI OFICIALES DEL II CORTE")
display(kpi_finales_gob)

print("\n6. RESUMEN GENERAL")
display(resumen_general_gob)

print("\n7. REGISTROS QUE REQUIEREN EXPLICACIÓN ADICIONAL")
display(
    excepciones[
        [
            "id_raw",
            "id_cita",
            "Motivos",
            "Estado_final"
        ]
    ].head(10)
)

print("\n8. EJEMPLOS DE REGISTROS APTOS CON HALLAZGOS")
display(
    aptos_con_hallazgos[
        [
            "id_raw",
            "id_cita",
            "Politicas_incumplidas",
            "Campos_normalizados",
            "Estado_final"
        ]
    ].head(10)
)

print("\n" + "=" * 70)
print("VERIFICACIÓN FINAL")
print("=" * 70)

print("Registros originales:", total_origen)
print("Registros aptos:", total_apto)
print("Registros en cuarentena:", total_cuarentena)
print("Copias excluidas:", total_copias)

print(f"Cobertura conservada: {cobertura:.2f}%")

print(
    f"Pruebas funcionales: "
    f"{pruebas_superadas}/{total_pruebas}"
)

print(
    f"KPI cumplidos: {kpi_cumplidos}/3"
)

print("Archivo depurado:", archivo_depurado.name)
print("Controles de consistencia: SUPERADOS")


# ============================================================
# 12. GUARDAR EVIDENCIAS FINALES
# ============================================================

archivos_finales = {
    "matriz_politicas_final": politicas_finales,
    "resumen_auditoria_final": auditoria_final_gob,
    "seguimiento_hallazgos": seguimiento_final,
    "pruebas_funcionales_final": pruebas_finales_gob,
    "kpi_oficiales_final": kpi_finales_gob,
    "resumen_general": resumen_general_gob,
    "registros_revision_adicional": excepciones,
    "aptos_con_hallazgos_iniciales": aptos_con_hallazgos
}

for nombre, tabla in archivos_finales.items():

    ruta = (
        carpeta_final_gob
        / f"{nombre}_{fecha_final_gob}.csv"
    )

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", ruta.name)


# ============================================================
# 13. CIERRE
# ============================================================

print("\nPASO 7 FINALIZADO")

print("Evidencias en:", carpeta_final_gob)

print("\nOBSERVACIONES:")
print("- POL-01 no pudo evaluarse por falta de fecha de cierre.")
print("- Los catálogos utilizados son provisionales.")
print("- Los 924 registros en cuarentena siguen en revisión.")
print("- Los KPI finales corresponden al conjunto apto.")
print("- El control no está integrado a una interfaz institucional.")
print("- SQL Server no fue modificado.")


RED SALUD INTEGRAL CARAZO
OPTATIVO IV - RESUMEN FINAL DE GOBIERNO DE DATOS

1. MATRIZ DE POLÍTICAS


,Codigo,Politica,Implementacion_tecnica,Prueba_funcional
0,POL-01,Cierre obligatorio de citas,NO EVALUABLE CON ESTOS DATOS,NO EJECUTADA
1,POL-02,Completitud de registros,PROGRAMADA,EJECUTADA
2,POL-03,Estandarización,PROGRAMADA,EJECUTADA
3,POL-04,Control de duplicidad,PROGRAMADA,EJECUTADA
4,POL-05,Control de valores anómalos,PROGRAMADA,EJECUTADA



2. AUDITORÍA POR POLÍTICA


,Politica,Incidencias,Registros_afectados
0,POL-02,920,920
1,POL-03,273,273
2,POL-04,254,160
3,POL-05,3,3



3. SEGUIMIENTO DE HALLAZGOS


...


7. REGISTROS QUE REQUIEREN EXPLICACIÓN ADICIONAL


,id_raw,id_cita,Motivos,Estado_final
5360,5361,RSV2-005361,EDA_COSTO_REQUIERE_REVISION,EN_CUARENTENA_PARA_REVISION



8. EJEMPLOS DE REGISTROS APTOS CON HALLAZGOS


,id_raw,id_cita,Politicas_incumplidas,Campos_normalizados,Estado_final
91,92,RSV2-000092,POL-04,0,APTO
101,102,RSV2-000102,POL-03,1,APTO
134,135,RSV2-000135,POL-03,1,APTO
167,168,RSV2-000168,POL-04,0,APTO
181,182,RSV2-000182,POL-03,1,APTO
199,200,RSV2-000200,POL-03,1,APTO
245,246,RSV2-000246,POL-03,1,APTO
257,258,RSV2-000258,POL-03,1,APTO
259,260,RSV2-000260,POL-03,1,APTO
292,293,RSV2-000293,POL-03,1,APTO



VERIFICACIÓN FINAL
Registros originales: 10080
Registros aptos: 9076
Registros en cuarentena: 924
Copias excluidas: 80
Cobertura conservada: 90.04%
Pruebas funcionales: 8/8
KPI cumplidos: 3/3
Archivo depurado: dataset_apto_20261024T210524Z.csv
Controles de consistencia: SUPERADOS
Guardado: matriz_politicas_final_20261024T213458Z.csv
Guardado: resumen_auditoria_final_20261024T213458Z.csv
Guardado: seguimiento_hallazgos_20261024T213458Z.csv
Guardado: pruebas_funcionales_final_20261024T213458Z.csv
Guardado: kpi_oficiales_final_20261024T213458Z.csv
Guardado: resumen_general_20261024T213458Z.csv
Guardado: registros_revision_adicional_20261024T213458Z.csv
Guardado: aptos_con_hallazgos_iniciales_20261024T213458Z.csv

PASO 7 FINALIZADO
Evidencias en: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\optativo_IV\resumen_final

OBSERVACIONES:
- POL-01 no pudo evaluarse por falta de fecha de cierre.
- Los catálogos utilizados son provisionales.
- Los 924 registros en cua